In [1]:
from bs4 import BeautifulSoup
import httpx

In [2]:
r = httpx.get("https://monome.org/docs/crow/reference/")
html = r.text

In [3]:
soup = BeautifulSoup(r.text)

In [4]:
tags = [tag.name for tag in soup.find_all()]

In [9]:
# tags

In [6]:
from llama_cpp.llama import Llama, LlamaGrammar
import httpx
from llama_cpp.llama import LlamaGrammar
# from PyPDF2 import PdfReader
import numpy as np
import pandas as pd
import torch
# from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.llms.llama_cpp import LlamaCPP
from llama_index.llms.llama_cpp.llama_utils import (
    messages_to_prompt,
    completion_to_prompt,
)
from llama_index.core import Settings
from llama_index.core import SimpleDirectoryReader, StorageContext
from llama_index.core import VectorStoreIndex
from llama_index.vector_stores.postgres import PGVectorStore
import textwrap
from llama_index.core import Document
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings
from transformers import AutoTokenizer
from llama_index.core import set_global_tokenizer

from llama_index.core.tools import QueryEngineTool, ToolMetadata
from llama_index.core.query_engine import RouterQueryEngine

/llm/.venv/lib/python3.11/site-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_url" in LlamaCPP has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/llm/.venv/lib/python3.11/site-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_path" in LlamaCPP has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/llm/.venv/lib/python3.11/site-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_kwargs" in LlamaCPP has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/llm/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.read

In [11]:
# html = SimpleDirectoryReader("/notebooks/llm/crow_rag/crowdocs/html/").load_data()

In [12]:
# reader = PdfReader('/notebooks/llm/pdf/Ableton12.pdf')

# pages = [page.extract_text() for page in reader.pages]
# pages = [i.encode('utf-8') for i in pages]
# docs = [Document(text=i) for i in pages]
# docs = [i for i in docs if i.get_content() != ""]

In [13]:
tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct")

llm = LlamaCPP(
    # You can pass in the URL to a GGML model to download it automatically
    # optionally, you can set the path to a pre-downloaded model instead of model_url
    # model_path="/hf_cache/models--NousResearch--Hermes-3-Llama-3.1-8B-GGUF/snapshots/307a5dfb59aa38d88b6cfd32f44b8ad7c1da9fb8/Hermes-3-Llama-3.1-8B.Q5_K_M.gguf",
    # model_url="https://huggingface.co/bartowski/DeepSeek-Coder-V2-Lite-Instruct-GGUF/blob/main/DeepSeek-Coder-V2-Lite-Instruct-Q6_K.gguf",
    model_path="/hf_cache/models--bartowski--DeepSeek-Coder-V2-Lite-Instruct-GGUF/snapshots/8f248fa2072348f77a8bc37754e470de1f61866e/DeepSeek-Coder-V2-Lite-Instruct-Q6_K.gguf",
    temperature=0.05,
    max_new_tokens=2048,
    context_window=int(16384*1.25),
    generate_kwargs={
        "repeat_penalty": 1.1,
        "top_k": 0,
        "top_p": 0
    },
    model_kwargs={
        "n_gpu_layers": -1,
        # "grammar": grammar
                 },
    # messages_to_prompt=messages_to_prompt,
    # completion_to_prompt=completion_to_prompt,
    verbose=True,
)

Settings.llm = llm

Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)

llama_model_loader: loaded meta data with 42 key-value pairs and 377 tensors from /hf_cache/models--bartowski--DeepSeek-Coder-V2-Lite-Instruct-GGUF/snapshots/8f248fa2072348f77a8bc37754e470de1f61866e/DeepSeek-Coder-V2-Lite-Instruct-Q6_K.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = deepseek2
llama_model_loader: - kv   1:                               general.name str              = DeepSeek-Coder-V2-Lite-Instruct
llama_model_loader: - kv   2:                      deepseek2.block_count u32              = 27
llama_model_loader: - kv   3:                   deepseek2.context_length u32              = 163840
llama_model_loader: - kv   4:                 deepseek2.embedding_length u32              = 2048
llama_model_loader: - kv   5:              deepseek2.feed_forward_length u32              = 10944
llama_model_loader:

In [14]:
vector_store = PGVectorStore.from_params(
    database='grover',
    host='postgres',
    password='grover',
    port=5432,
    user='grover',
    table_name="newest2",
    embed_dim=384,  
    hnsw_kwargs={
        "hnsw_m": 16,
        "hnsw_ef_construction": 64,
        "hnsw_ef_search": 40,
        "hnsw_dist_method": "vector_cosine_ops",
    },
)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

In [15]:
# By default, it will parse a select subset of HTML tags, but you can override this.

# The default tags are: ["p", "h1", "h2", "h3", "h4", "h5", "h6", "li", "b", "i", "u", "section"]


from llama_index.core.node_parser import HTMLNodeParser
# # tags = all_classes
parser = HTMLNodeParser(tags = [i for i in tags if len(i) < 3])

nodes = parser.get_nodes_from_documents(html)


In [16]:
# len(nodes)

In [17]:
# from llama_index.core.node_parser import SentenceSplitter
# Settings.embed_model = HuggingFaceEmbedding(
#     model_name="BAAI/bge-small-en-v1.5"
# )
# Settings.text_splitter = SentenceSplitter(chunk_size=768, chunk_overlap=30)

# # per-index
# index = VectorStoreIndex.from_documents(
#     [doc], storage_context=storage_context,
#     transformations=[SentenceSplitter(chunk_size=1024, chunk_overlap=30)], show_progress=True
# )

In [18]:
index = VectorStoreIndex(nodes, storage_context=storage_context, show_progress=True)

Generating embeddings: 100%|██████████| 334/334 [00:05<00:00, 62.86it/s] 


In [19]:
# index = VectorStoreIndex.from_documents(
#     [doc], storage_context=storage_context, show_progress=True
# )
# index = VectorStoreIndex.from_vector_store(vector_store=vector_store)


In [20]:
index

In [21]:
print(index.as_query_engine().query("Write a function for the crow device to create a LFO that slowly increases in speed over time. The function should take arguments to parameterize the rate of change and the output number."))


llama_print_timings:        load time =    1017.97 ms
llama_print_timings:      sample time =    1285.97 ms /   172 runs   (    7.48 ms per token,   133.75 tokens per second)
llama_print_timings: prompt eval time =    1017.04 ms /   223 tokens (    4.56 ms per token,   219.26 tokens per second)
llama_print_timings:        eval time =    6528.69 ms /   171 runs   (   38.18 ms per token,    26.19 tokens per second)
llama_print_timings:       total time =    8956.78 ms /   394 tokens



```python
import random

def create_crow_lfo(output_number, rate_of_change):
    # Initialize variables
    current_tempo = 120  # Starting tempo (BPM)
    increment = rate_of_change  # Rate of change for the tempo
    
    def update_tempo():
        nonlocal current_tempo
        current_tempo += increment
        return current_tempo
    
    while True:
        yield update_tempo()
```

This function initializes a Crow device LFO that starts at 120 BPM and increases its tempo by the specified rate of change. The `update_tempo` method increments the tempo, which is then yielded to simulate the changing tempo over time.


In [22]:
# vector_tool = QueryEngineTool(
#     index.as_query_engine(),
#     metadata=ToolMetadata(
#         name="vector_search",
#         description="Useful for searching for specific facts.",
#     ),
# )

# summary_tool = QueryEngineTool(
#     index.as_query_engine(response_mode="tree_summarize"),
#     metadata=ToolMetadata(
#         name="summary",
#         description="Useful for summarizing an entire document.",
#     ),
# )

In [23]:
# query_engine = RouterQueryEngine.from_defaults(
#     [vector_tool, summary_tool], select_multi=False, verbose=True
# )

# response = query_engine.query(
#     "What happened to Senator Hank Feltman?"
# )